# Agentic RAG Demo Notebook

This notebook demonstrates the final local agentic RAG prototype over uploaded PDF documents.

The codebase is written as a general PDF RAG system: any PDFs placed into `imports/` can be indexed and queried. For demonstration, the current corpus is the sample policy manual PDF placed in `imports/`.

The goal of this notebook is to show the major pipeline units, the retrieval strategy, the LangGraph orchestration logic, and the current system limitations.


## 1. Assignment Mapping

This notebook addresses the main requirements from the assignment:

- local Python implementation
- LangGraph-based orchestration
- RAG over external PDF documents
- agentic behavior through routing, query rewriting, broader retry search, answer verification, and smalltalk handling
- structured explanation of the architecture, technical choices, bottlenecks, and next steps

The current examples use the sample policy manual corpus, but the pipeline itself is designed for general PDF ingestion.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag.config import DATA_DIR, SOURCE_DOCUMENTS_DIR, EMBEDDING_MODEL_NAME, GENERATION_MODEL_NAME, RERANKER_MODEL_NAME
from rag.loaders import list_pdf_documents
from rag.retrieve import load_retrieval_model, load_reranker, retrieve_chunks, format_results, filter_results
from rag.graph import build_graph

print("Project root:", Path("."))
print("Import dir:", SOURCE_DOCUMENTS_DIR.relative_to(PROJECT_ROOT))
print("Indexed source candidates:", [path.relative_to(PROJECT_ROOT) for path in list_pdf_documents(SOURCE_DOCUMENTS_DIR)])
print("Data dir:", DATA_DIR.relative_to(PROJECT_ROOT))
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Generation model:", GENERATION_MODEL_NAME)
print("Reranker:", RERANKER_MODEL_NAME)


/home/nandor/Coding/PWC/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: .
Import dir: imports
Indexed source candidates: [PosixPath('imports/sample_policy_and_procedures_manual.pdf')]
Data dir: data
Embedding model: intfloat/e5-base-v2
Generation model: Qwen/Qwen2.5-3B-Instruct
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


## 2. Data and Pipeline Outputs

The project can ingest one or more PDFs from `imports/`. The preprocessing pipeline extracts and cleans page text, builds chunks, embeds them, and stores the vector index locally.

For the current demo, the imported corpus is the sample policy manual PDF.


In [2]:
artifacts = {
    "extracted_pages": DATA_DIR / "extracted_pages.json",
    "cleaned_pages": DATA_DIR / "cleaned_pages.json",
    "chunked_documents": DATA_DIR / "chunked_documents.json",
    "embedded_chunks": DATA_DIR / "embedded_chunks.json",
    "test_queries": DATA_DIR / "test_queries.json",
}

for name, path in artifacts.items():
    print(f"{name}: {path.relative_to(PROJECT_ROOT)}")


extracted_pages: data/extracted_pages.json
cleaned_pages: data/cleaned_pages.json
chunked_documents: data/chunked_documents.json
embedded_chunks: data/embedded_chunks.json
test_queries: data/test_queries.json


In [3]:
with open(DATA_DIR / "extracted_pages.json", "r", encoding="utf-8") as f:
    extracted_pages = json.load(f)

with open(DATA_DIR / "cleaned_pages.json", "r", encoding="utf-8") as f:
    cleaned_pages = json.load(f)

with open(DATA_DIR / "chunked_documents.json", "r", encoding="utf-8") as f:
    chunked_documents = json.load(f)

documents = sorted({page["document_name"] for page in extracted_pages})
print("Documents in current indexed corpus:", documents)
print("Extracted pages:", len(extracted_pages))
print("Cleaned pages:", len(cleaned_pages))
print("Chunks:", len(chunked_documents))


Documents in current indexed corpus: ['sample_policy_and_procedures_manual.pdf']
Extracted pages: 67
Cleaned pages: 62
Chunks: 127


## 3. Chunking Strategy

The final chunking strategy is not raw PDF block chunking. It uses page text extraction plus heading/list-aware recursive chunking.

Main ideas:

- remove repeated headers and footer noise
- filter obviously irrelevant pages such as title and table-of-contents pages
- preserve heading-like lines and policy lists
- split by semantic-ish structure before applying token limits

In [4]:
sample_chunk = chunked_documents[0]
sample_chunk

{'chunk_id': 'doc-1-p2-c1',
 'document_id': 'doc-1',
 'document_name': 'sample_policy_and_procedures_manual.pdf',
 'document_path': 'sample_policy_and_procedures_manual.pdf',
 'page_number': 2,
 'page_chunk_index': 1,
 'token_count': 288,
 'text': 'Introduction\n\nThe attached sample Policies and Procedures Manual was developed to assist community development corporations (CDCs) in their administration of federal funds. It includes sample personnel, accounting, financial management, procurement, and records management policies, and has two distinct purposes:\n\n\uf0b7 To provide emerging CDCs with sample policies and procedures so that they may be\n\nable to develop policies and procedures appropriate to their specific circumstances, and to provide their staff members with information regarding the type of systems that may be adopted in their administration of federal funds; and\n\n\uf0b7 To provide mature CDCs with sample policies and procedures to compare with their\n\nexisting manua

## 4. Retrieval Layer

The retrieval pipeline is hybrid:

- dense retrieval with `e5-base-v2`
- lexical retrieval with BM25
- reciprocal rank fusion
- reranking with a cross-encoder

This is a general retrieval setup rather than a corpus-specific ruleset. The current examples happen to use the sample policy manual corpus.


In [5]:
query = "What is the holiday policy?"
retrieval_model = load_retrieval_model()
reranker = load_reranker()
raw_results = retrieve_chunks(query, retrieval_model, top_k=4, use_reranker=True, reranker=reranker)
results = filter_results(format_results(raw_results))
[(item['metadata']['page_number'], item['metadata']['page_chunk_index']) for item in results]

[(24, 4), (23, 3), (23, 2), (13, 3)]

In [6]:
print(results[0]['text'])

Holidays

CDC recognizes the following paid holidays:

1. New Year’s Day 6. Labor Day 2. Martin Luther King, Jr. Day 7. Columbus Day 3. President’s Day 8. Veteran’s Day 4. Memorial Day 9. Thanksgiving 5. Independence Day 10. Christmas Day

On National Election Day CDC allows all staff up to hours time off to vote, or time off as required by local law An employee must notify his/her supervisor in advance when time off to vote is to be used. Leave of Absence (non FMLA) Employees are eligible to apply for an unpaid leave of absence if they have been a regular employee for at least one year and scheduled to work hours or more per week. Employees requesting an unpaid leave of absence must do so in writing at least days in advance unless it is impractical to do so, in which case the employee must provide as much written notice as possible. The request must be submitted to the Executive Director.


## 5. Agentic Workflow

The orchestration layer uses LangGraph.

Current decision stages:

1. classify the input as question or smalltalk
2. detect follow-up questions
3. rewrite contextual follow-up queries
4. run primary hybrid retrieval
5. optionally broaden search if retrieval looks weak
6. generate an answer from retrieved context
7. verify the answer and fall back to extractive output if needed
8. short-circuit smalltalk acknowledgements without retrieval


In [7]:
graph = build_graph()
result = graph.invoke({"question": "What is the holiday policy?", "chat_history": []})
result['answer']

'The holiday policy includes the following:\n\n- Paid holidays: New Year’s Day, Martin Luther King, Jr. Day, President’s Day, Memorial Day, Independence Day, and Christmas Day.\n- Unpaid leave of absence: Allowed on National Election Day for up to hours, with a requirement to notify the supervisor in advance.\n- Vacation days: Accumulated based on the length of employment and can be taken as vacation time if an authorized holiday occurs during the period. Vacation days not taken can be carried over to the next year, but any unused days at the end of the year are forfeited. Vacation is taken in full day or half day increments.'

In [8]:
result['agent_trace']

['classify_question: intent=question, broad_search=True, follow_up=False',
 'rewrite_query: original',
 'retrieve_primary: retrieved=4',
 'assess_retrieval: retry=False',
 'generate_answer: model=Qwen/Qwen2.5-3B-Instruct',
 'verify_answer: accepted']

## 6. Follow-up Behavior

The system also supports short contextual follow-up questions by rewriting them against recent user history.

In [9]:
history = [
    {"role": "user", "content": "What is the holiday policy?"},
    {"role": "assistant", "content": result['answer']},
]
follow_up = graph.invoke({"question": "And what about vacation?", "chat_history": history})
follow_up['answer']

'During the second year of employment and each year thereafter, full-time employees accrue vacation day credit at the rate of days per month - weeks vacation each year. Any accumulated vacation over days at year-end is forfeited. Vacation time may be taken in full day or half day increments. Vacation days can be carried over from one calendar year to the next calendar year, but not exceeding days. Employees may take up to a maximum of weeks vacation at one time, subject to the approval of the supervisor. Vacation time is based on the base period of the anniversary date of employment. Vacation may be taken for illness in excess of the time allowed under sick day accrual, and such use of vacation time requires approval from the immediate supervisor.'

In [10]:
follow_up['agent_trace']

['classify_question: intent=question, broad_search=False, follow_up=True',
 'rewrite_query: contextualized_follow_up',
 'retrieve_primary: retrieved=4',
 'assess_retrieval: retry=False',
 'generate_answer: model=Qwen/Qwen2.5-3B-Instruct',
 'verify_answer: accepted']

## 7. Smalltalk Routing

To demonstrate that the system does not always retrieve blindly, short acknowledgements are routed away from retrieval.

In [11]:
smalltalk = graph.invoke({"question": "cool", "chat_history": history})
smalltalk['answer'], smalltalk['agent_trace']

("Understood. Ask the next question when you're ready.",
 ['classify_question: intent=smalltalk, broad_search=False, follow_up=False',
  'respond_smalltalk: no_retrieval'])

## 8. Evaluation Queries

A small fixed evaluation suite is included to sanity-check retrieval behavior on the current demo corpus.

Important:

- the evaluation query file is corpus-specific
- the included queries target the sample policy manual currently stored in `imports/`
- if a different set of PDFs is indexed, the query set should also be updated


In [12]:
with open(DATA_DIR / "test_queries.json", "r", encoding="utf-8") as f:
    test_queries = json.load(f)

test_queries[:8]

[{'id': 'holiday_policy',
  'category': 'benefits',
  'query': 'What is the holiday policy?'},
 {'id': 'vacation_rules',
  'category': 'benefits',
  'query': 'What are the vacation rules?'},
 {'id': 'grievance_procedure',
  'category': 'personnel',
  'query': 'How does the grievance procedure work?'},
 {'id': 'harassment_policy',
  'category': 'personnel',
  'query': 'What does the harassment policy say?'},
 {'id': 'direct_deposit',
  'category': 'personnel',
  'query': 'How are employees paid and what is direct deposit?'},
 {'id': 'conflict_of_interest',
  'category': 'personnel',
  'query': 'What is the conflict of interest policy?'},
 {'id': 'travel_reimbursement',
  'category': 'personnel',
  'query': 'What travel expenses are reimbursed?'},
 {'id': 'family_medical_leave',
  'category': 'benefits',
  'query': 'What does the family and medical leave policy say?'}]

## 9. Bottlenecks and Limitations

Current bottlenecks:

- local answer generation is the slowest part of the system
- GPU availability is critical for acceptable response times
- larger local models caused instability on the available hardware
- some questions still require extractive fallback instead of full natural generation

Current limitations:

- the fixed evaluation suite is corpus-specific
- retrieval is often stronger than the final generated answer
- abstract questions are less stable than direct factual questions
- adjacent chunks can still be blended into one answer when the context is semantically close
- placeholder-like source wording can still leak into generated answers
- larger and more mixed multi-document corpora still need broader testing

Most likely bottlenecks:

- latency bottleneck: local hardware and GPU availability
- answer-quality bottleneck: the smaller local generation model rather than the retrieval stack itself


## 10. Next Steps

Planned improvements:

- stronger evaluation with expected-document and expected-page labels
- stricter answer verification for weak or overconfident generations
- more structured answer modes for list, threshold, and definition-style questions
- cleaner and smaller generation context to reduce cross-section mixing
- shorter, more factual prompting for local generation
- cleaner answer formatting for extractive fallback
- more robust local model serving in a dedicated process
- broader mixed-corpus testing with multiple unrelated PDFs
